In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, max, min, round, when, count

spark = SparkSession.builder \
    .appName("Tarea 1 - FIFA Players") \
    .getOrCreate()

In [ ]:
df_players = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/players_22.csv")

In [ ]:
df_players.show(5)
df_players.printSchema()

In [ ]:
df_players.select(
    "overall", "potential", "age", "height_cm", "weight_kg",
    "pace", "shooting", "passing", "dribbling", "defending", "physic"
).describe().show()

In [ ]:
jugadores_elite = df_players.filter(col("overall") >= 85) \
    .select("short_name", "club_name", "nationality_name", "overall", "potential", "age")

jugadores_elite.orderBy(col("overall").desc()).show(10, truncate=False)

In [ ]:
mexicanos = df_players.filter(col("nationality_name") == "Mexico") \
    .select("short_name", "club_name", "overall", "potential", "player_positions", "age")

mexicanos.orderBy(col("overall").desc()).show(15, truncate=False)

In [ ]:
mbappe = df_players.filter(col("short_name").contains("Mbapp"))
mbappe.select(
    "short_name", "club_name", "nationality_name", "overall", "potential",
    "pace", "shooting", "passing", "dribbling", "defending", "physic"
).show(truncate=False)

In [ ]:
df_con_ipc = df_players.withColumn(
    "IPC",
    round(
        (col("potential") - col("overall")) / (100 - col("overall")) * 100,
        2
    )
)

promesas = df_con_ipc.filter((col("age") <= 23) & (col("overall") >= 70)) \
    .select("short_name", "age", "overall", "potential", "IPC", "club_name", "nationality_name")

promesas.orderBy(col("IPC").desc()).show(10, truncate=False)

In [ ]:
df_scores = df_players.withColumn(
    "score_ofensivo",
    round((col("pace") + col("shooting") + col("dribbling")) / 3, 2)
).withColumn(
    "score_defensivo",
    round((col("defending") + col("physic")) / 2, 2)
).withColumn(
    "perfil",
    when(col("score_ofensivo") > col("score_defensivo"), "Ofensivo")
    .otherwise("Defensivo")
)

df_scores.select(
    "short_name", "player_positions", "overall",
    "score_ofensivo", "score_defensivo", "perfil"
).orderBy(col("overall").desc()).show(15, truncate=False)

In [ ]:
por_pais = df_players.groupBy("nationality_name").agg(
    count("*").alias("num_jugadores"),
    round(avg("overall"), 2).alias("promedio_overall"),
    round(avg("potential"), 2).alias("promedio_potencial"),
    max("overall").alias("mejor_overall")
)

por_pais.orderBy(col("num_jugadores").desc()).show(10, truncate=False)